# LLM Baseline Evaluation (API-Based)

Evaluates commercial LLMs on the SVD-Benchmark using their public APIs.

| Model | Provider | API |
|-------|----------|-----|
| ChatGPT-5.2 | OpenAI | openai Python SDK |
| Claude 4.5 Sonnet | Anthropic | anthropic Python SDK |
| Gemini 2.5 Flash | Google | google-generativeai SDK |

**Paper methodology:**
- Same Alpaca-style instruction-input-output prompt (Table 2) used across all models.
- Zero-shot prompting only — no few-shot examples or chain-of-thought.
- Evaluated using official public interfaces (default system configuration).

> **Note:** The paper states LLMs were evaluated through their official web interfaces.
> This notebook implements the equivalent API-based evaluation for full automation
> and reproducibility. Results may differ slightly from manual web-interface runs.


In [ ]:
# !pip install openai anthropic google-generativeai scikit-learn tqdm --quiet
import os, time, pandas as pd, sys
from tqdm import tqdm

# Add repo root to path for shared utilities
sys.path.insert(0, ".")
from metrics_utils import parse_model_output, compute_all_metrics

# ── API Keys ────────────────────────────────────────────────────────────────
# Set these as environment variables before running:
#   export OPENAI_API_KEY="sk-..."
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   export GOOGLE_API_KEY="..."
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
GOOGLE_API_KEY    = os.getenv("GOOGLE_API_KEY", "")


## 1. Shared Zero-Shot Prompt

Identical Alpaca format used for VulnLocAI (Table 2). The LLMs receive the same instruction without any CWE ID hint.

In [ ]:
INSTRUCTION = "Analyze the following Java code snippet and identify any security vulnerability"

SYSTEM_PROMPT = (
    "You are a Java security expert. When given a code snippet, identify any security "
    "vulnerability. If vulnerable, respond in EXACTLY this format: "
    "'CWE-XX, <exact vulnerable line>, <brief description>'. "
    "If no vulnerability exists, respond: 'No vulnerability detected. The code is secure.'"
)

def build_user_message(code_snippet: str) -> str:
    return (
        f"### Instruction:\n{INSTRUCTION}\n\n"
        f"### Input:\n{code_snippet}\n\n"
        f"### Response:"
    )


## 2. API Wrapper Functions

Each function follows the zero-shot strategy from.

In [ ]:
# ── OpenAI (ChatGPT-5.2) ────────────────────────────────────────────────────
import openai

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

def predict_chatgpt(code_snippet: str, model: str = "gpt-4.5-preview") -> str:
    """Zero-shot prediction using OpenAI API."""
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system",  "content": SYSTEM_PROMPT},
                {"role": "user",    "content": build_user_message(code_snippet)},
            ],
            max_tokens=256,
            temperature=0,      # deterministic (greedy equivalent)
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"ERROR: {e}"


# ── Anthropic (Claude 4.5 Sonnet) ───────────────────────────────────────────
import anthropic

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def predict_claude(code_snippet: str, model: str = "claude-sonnet-4-5") -> str:
    """Zero-shot prediction using Anthropic API."""
    try:
        message = anthropic_client.messages.create(
            model=model,
            max_tokens=256,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": build_user_message(code_snippet)}],
        )
        return message.content[0].text.strip()
    except Exception as e:
        return f"ERROR: {e}"


# ── Google (Gemini 2.5 Flash) ────────────────────────────────────────────────
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    system_instruction=SYSTEM_PROMPT,
)

def predict_gemini(code_snippet: str) -> str:
    """Zero-shot prediction using Google Gemini API."""
    try:
        response = gemini_model.generate_content(
            build_user_message(code_snippet),
            generation_config=genai.GenerationConfig(
                max_output_tokens=256,
                temperature=0,
            ),
        )
        return response.text.strip()
    except Exception as e:
        return f"ERROR: {e}"


## 3. Load SVD-Benchmark

In [ ]:
benchmark_df = pd.read_csv("Evaluation-Benchmark/SVD-Benchmark.csv")
print(f"Loaded {len(benchmark_df)} samples")


## 4. Run Evaluations

A 0.5-second delay between API calls avoids rate limiting. Adjust `RATE_LIMIT_DELAY` if needed.

In [ ]:
RATE_LIMIT_DELAY = 0.5   # seconds between API calls

LLM_CONFIGS = [
    ("ChatGPT-5.2",        predict_chatgpt,  bool(OPENAI_API_KEY)),
    ("Claude 4.5 Sonnet",  predict_claude,   bool(ANTHROPIC_API_KEY)),
    ("Gemini 2.5 Flash",   predict_gemini,   bool(GOOGLE_API_KEY)),
]

all_results = {}

for name, predict_fn, has_key in LLM_CONFIGS:
    if not has_key:
        print(f"Skipping {name}: API key not set.")
        continue

    print(f"\n{'='*60}")
    print(f"  Evaluating: {name}")
    print(f"{'='*60}")

    predictions, raw_col = [], []
    for _, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df), desc=name):
        raw = predict_fn(str(row["Code Snippet"]))
        predictions.append(parse_model_output(raw))
        raw_col.append(raw)
        time.sleep(RATE_LIMIT_DELAY)

    col = name.replace(" ", "_")
    benchmark_df[f"{col}_raw"]       = raw_col
    benchmark_df[f"{col}_pred_label"]= [p["predicted_label"] for p in predictions]
    benchmark_df[f"{col}_pred_line"] = [p["predicted_line"]  for p in predictions]

    all_results[name] = compute_all_metrics(predictions, benchmark_df, model_name=name)


## 5. Summary Table

In [ ]:
if all_results:
    summary = pd.DataFrame(list(all_results.values()))
    print(summary.to_string(index=False))
    summary.to_csv("Results/LLM_baselines_metrics.csv", index=False)
    benchmark_df.to_csv("Results/LLM_baselines_predictions.csv", index=False)
    print("Saved to Results/")
